In [0]:
USE CATALOG workspace;
USE SCHEMA ecom_poc;

In [0]:
CREATE OR REPLACE TABLE bronze_orders
COMMENT 'Raw orders, all columns as STRING, loaded unchanged from orders.csv'
AS
SELECT
  order_id, customer_id, order_date, category,
  quantity, unit_price, order_status, country,
  current_timestamp()          AS _ingested_at,
  _metadata.file_path          AS _source_file,
  'orders_batch_20261118_001'  AS _batch_id,
  sha2(concat_ws('|', order_id, customer_id, order_date, category,
                 quantity, unit_price, order_status, country), 256) AS _row_hash
FROM read_files(
  '/Volumes/workspace/ecom_poc/raw_files/orders.csv',
  format    => 'csv',
  header    => true,
  schema    => 'order_id STRING, customer_id STRING, order_date STRING, category STRING, quantity STRING, unit_price STRING, order_status STRING, country STRING',
  nullValue => '__NO_NULL__'
);

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bronze_transactions
COMMENT 'Raw transactions, all columns as STRING, loaded unchanged from transactions.csv'
AS
SELECT
  txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts,
  current_timestamp()                AS _ingested_at,
  _metadata.file_path                AS _source_file,
  'transactions_batch_20261118_001'  AS _batch_id,
  sha2(concat_ws('|', txn_id, order_id, payment_method, amount,
                 currency, txn_status, txn_ts), 256) AS _row_hash
FROM read_files(
  '/Volumes/workspace/ecom_poc/raw_files/transactions.csv',
  format    => 'csv',
  header    => true,
  schema    => 'txn_id STRING, order_id STRING, payment_method STRING, amount STRING, currency STRING, txn_status STRING, txn_ts STRING',
  nullValue => '__NO_NULL__'
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT 'orders' AS dataset,
       (SELECT COUNT(*) FROM read_files('/Volumes/workspace/ecom_poc/raw_files/orders.csv',
            format => 'csv', header => true)) AS source_rows,
       (SELECT COUNT(*) FROM bronze_orders) AS bronze_rows
UNION ALL
SELECT 'transactions',
       (SELECT COUNT(*) FROM read_files('/Volumes/workspace/ecom_poc/raw_files/transactions.csv',
            format => 'csv', header => true)),
       (SELECT COUNT(*) FROM bronze_transactions);

dataset,source_rows,bronze_rows
orders,1000000,1000000
transactions,1000000,1000000


In [0]:
DESCRIBE TABLE bronze_orders;
SELECT * FROM bronze_orders LIMIT 10;
DESCRIBE TABLE bronze_transactions;
SELECT * FROM bronze_transactions LIMIT 10;

txn_id,order_id,payment_method,amount,currency,txn_status,txn_ts,_ingested_at,_source_file,_batch_id,_row_hash
TXN-0635081,ORD-0735141,CARD,34.21,USD,success,2025-07-30 20:53:06,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,e26260aca31a5b3209b8746590fc3ba96618762159e9af5cfca3e4c47c22604f
TXN-0286514,ORD-0336141,CARD,666.92,USD,SUCCESS,2026-08-23 07:29:03,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,061ef3502a232d37e91f5fbb8ebff2ccaedace8964ed1bd54057ba0378aed573
TXN-0900567,ORD-0144339,PAYPAL,149.20,USD,SUCCESS,2026-10-29 11:58:57,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,6fe243e74f582536887d3416c6b3de90f493371770d8bf3d8ab0ea762aa708f8
TXN-0038385,ORD-0329822,CARD,18.42,USD,SUCCESS,2025-01-01 12:41:32,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,77a86e1973828985a35902a53d1900e167af58a80342e9ab471d6a4414d4673d
TXN-0572418,ORD-0478884,WALLET,30.64,GBP,SUCCESS,2025-03-12 22:20:01,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,04e196479754dd3339f6294cecad5c4c353a9a4b757a262d44131b12a422d0f5
TXN-0923605,ORD-0850412,PAYPAL,9.05,USD,REFUNDED,2026-05-10 21:26:11,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,88efa6906b1cefed029dd5abfaf7fa05e90d963e9575e33e6e4361cb87273ce0
TXN-0489533,ORD-0431937,WALLET,193.12,USD,SUCCESS,2026-01-11 20:17:29,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,20473909656b8fd1d8c2b107a2424b8913ddc01b60d0657284d91b24ddfa6a6e
TXN-0503435,ORD-0299907,BANK_TRANSFER,65.50,USD,SUCCESS,2025-02-30 10:00:00,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,af889586f68868398fc43a84f787a3874a9e7d6cadcf04d86abb6bad1eccb6e4
TXN-0876536,ORD-0806542,BANK_TRANSFER,106.14,USD,SUCCESS,2025-12-17 19:16:31,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,66378a80e659109a946889eb1b0884d19fd6748b550f532738fc578d9b51b7e3
TXN-0162282,ORD-0815675,CARD,15.07,USD,PENDING,2025-08-23 05:38:47,2026-11-18T12:28:30.443Z,dbfs:/Volumes/workspace/ecom_poc/raw_files/transactions.csv,transactions_batch_20261118_001,7d9e52bc7cdc6b43ace48838dbf965657801450db8324d693d62c9731f3f679f
